In [1]:
import pandas as pd
from ete3 import Tree
import os
pd.set_option('display.max_rows', 100)  # Replace 100 with the desired number of rows

In [ ]:
from pathlib import Path

# Set up project paths
project_root = Path('/workspaces/CellTreeBench')
data_dir = project_root / 'data' / 'celegans_small' / 'P0' / 'tree_building' / 'tree_df_adjusted'
out_dir = project_root / 'data' / 'celegans_small' / 'P0'

# Ensure output directory exists
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Input directory: {data_dir}")
print(f"Output directory: {out_dir}")

In [3]:
new_rows = [
    {'Lineage': 'P0', 'Parent': "", 'n_cells': 0},
    {'Lineage': 'AB', 'Parent': 'P0', 'n_cells': 0},
    {'Lineage': 'P1', 'Parent': 'P0', 'n_cells': 0},
    {'Lineage': 'ABPx', 'Parent': 'AB', 'n_cells': 0},
    {'Lineage': 'EMS', 'Parent': 'P1', 'n_cells': 0},
    {'Lineage': 'P2', 'Parent': 'P1', 'n_cells': 0},
]
tree_df = pd.DataFrame(new_rows)
tree_df

,Lineage,Parent,n_cells
0,P0,,0
1,AB,P0,0
2,P1,P0,0
3,ABPx,AB,0
4,EMS,P1,0
5,P2,P1,0


In [4]:
def create_tree(tree_df):
    # Create the root of the tree
    node_dict = {}
    root_name = tree_df["Lineage"].iloc[0]
    tree = Tree(name=root_name)
    tree.add_feature("n_cells", tree_df["n_cells"].iloc[0])

    node_dict[root_name] = tree

    # Create nodes and arrange by parent
    for index, row in tree_df.iterrows():
        lineage_name = row["Lineage"]
        parent_name = row["Parent"]
        # Skip nodes with n_cells = 0
        # if row["n_cells"] == 0 or lineage_name == root_name:
        if lineage_name == root_name:
            continue

        # Add current node if it doesn't exist
        if lineage_name not in node_dict:
            node = Tree(name=lineage_name)
            node.add_feature("n_cells", row["n_cells"])
            node_dict[lineage_name] = node

        # Ensure parent node exists
        if parent_name not in node_dict:
            # node_dict[parent_name] = Tree(name=parent_name)
            print(f"ERR: Parent node {parent_name} not found for {lineage_name}.")

        # Attach the current node to its parent
        node_dict[parent_name].add_child(node_dict[lineage_name])
    return tree

In [5]:
hat_tree = create_tree(tree_df)
print(hat_tree.get_ascii(attributes=["name"]))


   /AB/-ABPx
-P0
  |   /-EMS
   \P1
      \-P2


In [6]:
subtree_root_dict = {
    "ABaxx": "AB",
    "ABpxp": "ABPx",
    "ABpxax": "ABPx",
    "MSx": "EMS",
    "Exx": "EMS",
    "Cx": "P2",
    "Dx": "P2",
}

In [ ]:
processed_subtrees = {}

for lineage_name in subtree_root_dict.keys():
    print(f"Processing {lineage_name}")
    file_name = data_dir / f"tree_df-{lineage_name}.csv"
    
    if not file_name.exists():
        print(f"Warning: File not found: {file_name}")
        continue
        
    the_df = pd.read_csv(file_name)
    print(f"  Loaded {len(the_df)} nodes for {lineage_name}")
    
    # Update parent for root node
    the_df.loc[the_df["Lineage"] == lineage_name, "Parent"] = subtree_root_dict[lineage_name]
    the_df = the_df[['Lineage', 'Parent', 'n_cells']]
    
    # Count leaves before merging
    temp_tree = create_tree(the_df)
    n_leaves = len(temp_tree.get_leaf_names())
    processed_subtrees[lineage_name] = n_leaves
    print(f"  {lineage_name} contributes {n_leaves} leaves")
    
    tree_df = pd.concat([tree_df, the_df], ignore_index=True)

print(f"\nSubtree processing summary:")
for lineage, n_leaves in processed_subtrees.items():
    print(f"  {lineage}: {n_leaves} leaves")
print(f"Total nodes in merged tree: {len(tree_df)}")

Processing ABaxx
Processing ABpxp
Processing ABpxax
Processing MSx
Processing Exx
Processing Cx
Processing Dx


In [8]:
tree_df

,Lineage,Parent,n_cells
0,P0,,0
1,AB,P0,0
2,P1,P0,0
3,ABPx,AB,0
4,EMS,P1,0
...,...,...,...
257,Dxxax,Dxxa,134
258,Dxppa,Dxxp,90
259,Dxppp,Dxxp,111
260,Dxppax,Dxppa,123


In [ ]:
# Create final merged tree
the_tree = create_tree(tree_df)
n_leaves = len(the_tree.get_leaf_names())
print(f"Final merged tree has {n_leaves} leaves")

tree_name = "P0"

# Save ASCII representation
file_name = out_dir / f"{tree_name}.txt"
with open(file_name, "w") as f:
    f.write(the_tree.get_ascii(attributes=["name"]))
print(f"Saved tree ASCII to: {file_name}")

# Save ASCII with cell counts
file_name = out_dir / f"{tree_name}-ncells.txt"
with open(file_name, "w") as f:
    f.write(the_tree.get_ascii(attributes=["name", "n_cells"]))
print(f"Saved tree ASCII with cell counts to: {file_name}")

# Save final tree dataframe (this is the key file for the dataset class)
file_name = out_dir / f"tree_df-{tree_name}.csv"
tree_df.to_csv(file_name, index=False)
print(f"Saved final tree dataframe to: {file_name}")

print(f"\n{'='*60}")
print("TREE MERGING COMPLETE!")
print(f"{'='*60}")
print(f"Final tree statistics:")
print(f"  Total nodes: {len(tree_df)}")
print(f"  Leaf nodes: {n_leaves}")
print(f"  Tree depth: {the_tree.get_farthest_leaf()[1]}")
print(f"\nKey output file for dataset class:")
print(f"  {file_name}")
print(f"\nAll files saved to: {out_dir}")    

number of children: 103
